# Clase 15: MLOps, Docker y FastAPI

**Machine Learning (RIA-805) — Bloque 3 — Formulación de Proyectos y Emprendimiento Sostenible**

**Docente:** [tu nombre]  
**Fecha:** Semana 5

**Objetivo:** Comprender los fundamentos de MLOps, exportar un modelo a ONNX, construir una API REST con FastAPI y Pydantic, dockerizarla y medir su latencia.

**Conexión con la práctica:** En P3-1 a P3-5 harán exactamente esto con su modelo de Wall-Following. Hoy verán el proceso completo.

**Nota:** La Clase 14 (relevancia y sostenibilidad) se convirtió en AA 3.2. Aquí solo hacemos una introducción breve.

## Introducción a relevancia y sostenibilidad (20 min)

Antes de desplegar un modelo, hay que explicar por qué importa y a quién beneficia. La **relevancia** se puede analizar con seis preguntas: ¿qué problema concreto se atiende?, ¿quiénes son las personas afectadas?, ¿qué evidencia respalda la necesidad?, ¿qué alternativas existen?, ¿qué consecuencias no deseadas pueden aparecer? y ¿cómo se medirá el beneficio?

La **sostenibilidad** considera tres dimensiones conectadas: ambiental (energía y materiales), social (equidad, accesibilidad y efectos en las comunidades) y económica (costos, mantenimiento y viabilidad a largo plazo). Un framework práctico tiene siete pasos: definir el contexto; identificar actores; establecer una línea base; formular objetivos e indicadores; estimar impactos y riesgos; diseñar mitigaciones; y revisar resultados con transparencia.

**Green AI** promueve medir y reducir el costo computacional y energético durante entrenamiento e inferencia, sin ocultar las limitaciones de las estimaciones. En AA 3.2 aplicarán esto a su proyecto de P3.

## ¿Qué es MLOps?

**MLOps** reúne prácticas para desarrollar, desplegar y mantener sistemas de aprendizaje automático de forma reproducible. DevOps coordina el ciclo de vida del software; MLOps incorpora además datos, experimentos, modelos, deriva y reentrenamiento. No es solo publicar un endpoint: también hay que versionar y monitorear lo que cambia.

```text
datos → validación → entrenamiento → evaluación → despliegue
   ↑                                           ↓
   └──────── monitoreo ← predicciones ← servicio
                     → reentrenamiento
```

| Componente | Propósito |
|---|---|
| Versionado de datos | Saber qué datos produjeron cada resultado |
| Versionado de modelos | Identificar y recuperar artefactos |
| Empaquetado | Llevar modelo y preprocesamiento juntos |
| Serving | Exponer predicciones mediante una interfaz |
| Contenerización | Aislar el entorno y sus dependencias |
| Orquestación | Coordinar procesos y recursos |
| Monitoreo | Medir salud, calidad, latencia y deriva |
| CI/CD | Probar y publicar cambios de manera controlada |

En P3 verán: empaquetado (ONNX), serving (FastAPI), contenerización (Docker), profiling y Green AI.

## ONNX: el formato estándar

**ONNX (Open Neural Network Exchange)** es un formato abierto para representar modelos de aprendizaje automático. Un modelo exportado puede ejecutarse con distintos runtimes sin cargar el estimador original de scikit-learn.

Ventajas: portabilidad entre herramientas y lenguajes, interoperabilidad y, según el hardware y el modelo, potenciales mejoras de rendimiento. Exportar no garantiza equivalencia ni aceleración: hay que comparar predicciones, latencia y tamaño.

Ejemplo conceptual de scikit-learn:

```python
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

onnx_model = convert_sklearn(
    pipeline,
    initial_types=[("input", FloatTensorType([None, n_features]))],
)
```

En esta clase exportamos tanto el escalado como el clasificador, de modo que el servicio reciba las siete mediciones originales.

In [1]:
# Importaciones comunes y configuración reproducible del notebook.
import json
import os
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

%matplotlib inline
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Ubicamos la raíz del proyecto aunque Jupyter se haya iniciado desde notebooks/.
PROJECT_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "processed" / "crop.parquet").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontró data/processed/crop.parquet en el directorio actual ni en sus padres."
    )
os.chdir(PROJECT_ROOT)
(PROJECT_ROOT / "app").mkdir(parents=True, exist_ok=True)
# app.py en la raíz es un módulo Streamlit; __init__.py hace de app/ un paquete importable por Uvicorn.
(PROJECT_ROOT / "app" / "__init__.py").touch(exist_ok=True)
(PROJECT_ROOT / "models").mkdir(parents=True, exist_ok=True)
print(f"Raíz del proyecto: {PROJECT_ROOT}")

# Las dependencias opcionales se detectan sin bloquear las secciones teóricas.
try:
    import onnx
    import onnxruntime as ort
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    ONNX_AVAILABLE = True
except ImportError as exc:
    ONNX_AVAILABLE = False
    print(f"Advertencia: faltan dependencias ONNX ({exc}). Instala: pip install onnx onnxruntime skl2onnx")

try:
    from fastapi import FastAPI
    from pydantic import BaseModel, Field
    import uvicorn
    FASTAPI_AVAILABLE = True
except ImportError as exc:
    FASTAPI_AVAILABLE = False
    print(f"Advertencia: faltan dependencias de API ({exc}). Instala: pip install fastapi uvicorn pydantic")

try:
    import psutil
    PSUTIL_AVAILABLE = True
except ImportError:
    PSUTIL_AVAILABLE = False
    print("Advertencia: psutil no está disponible; la lectura de RAM se omitirá.")

# Optuna no es requisito funcional para esta sesión, pero se detecta para el entorno de curso.
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ImportError:
    print("Aviso: Optuna no está instalado; no se necesita para ejecutar este notebook.")

Raíz del proyecto: c:\Users\marco\ml-demo1-profesor


## Cargar modelo y datos de B2

Vamos a usar el modelo final del Bloque 2 (`models/rf_final_b2.pkl`) y los datos procesados. Ese archivo contiene el clasificador, pero no el escalador. La API no debe recibir valores crudos y pasarlos directamente al clasificador: aquí reconstruiremos el `RobustScaler` reproduciendo las particiones con `random_state=42` de la Clase 13, y empaquetaremos scaler y modelo juntos.

Si el modelo binario no está disponible, se carga `models/random_forest.pkl` como alternativa de demostración. Ese artefacto histórico es multiclase y **no** es equivalente al modelo binario rice/no-rice; el notebook conserva explícitamente esa distinción.

In [2]:
# Cargamos datos, metadatos y el modelo disponible.
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "crop.parquet"
MODEL_PATH = PROJECT_ROOT / "models" / "rf_final_b2.pkl"
LEGACY_MODEL_PATH = PROJECT_ROOT / "models" / "random_forest.pkl"
CONFIG_PATH = PROJECT_ROOT / "config.json"
BINARY_MODEL = MODEL_PATH.exists()

if BINARY_MODEL:
    model = joblib.load(MODEL_PATH)
    config = json.loads(CONFIG_PATH.read_text(encoding="utf-8")) if CONFIG_PATH.exists() else {}
    print(f"Modelo binario B2 cargado: {MODEL_PATH}")
elif LEGACY_MODEL_PATH.exists():
    model = joblib.load(LEGACY_MODEL_PATH)
    config = {}
    print("Advertencia: usando el modelo histórico multiclase; no representa rice vs no-rice.")
else:
    raise FileNotFoundError("No se encontró rf_final_b2.pkl ni random_forest.pkl en models/.")

df = pd.read_parquet(DATA_PATH)
feature_columns = config.get(
    "feature_columns",
    ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"],
)
if "label" not in df.columns or not set(feature_columns).issubset(df.columns):
    raise ValueError("El parquet debe contener label y las siete features definidas en config.json.")
X_raw = df[feature_columns].copy()
y = (df["label"].astype(str).str.lower() == "rice").astype(int)

# Reproducimos el split anidado de la Clase 13: el scaler se ajusta solo en train (64%).
X_trainval, X_test_raw, y_trainval, y_test = train_test_split(
    X_raw, y, test_size=0.20, stratify=y, random_state=42
)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.20, stratify=y_trainval, random_state=42
)

if BINARY_MODEL:
    scaler = RobustScaler()
    scaler.fit(X_train_raw)
    serving_model = Pipeline([("scaler", scaler), ("classifier", model)])
    threshold = float(config.get("threshold", 0.5))
    mode = "binary"
    # La envoltura incluye exactamente el preprocesamiento que esperaba el modelo B2.
    X_test_model = scaler.transform(X_test_raw).astype(np.float32)
else:
    serving_model = model
    threshold = 0.5
    mode = "multiclass"
    X_test_model = X_test_raw.to_numpy(dtype=np.float32)

MODEL_BUNDLE_PATH = PROJECT_ROOT / "models" / "rf_final_b2_pipeline.pkl"
joblib.dump(serving_model, MODEL_BUNDLE_PATH)
config.update({
    "feature_columns": list(feature_columns),
    "threshold": threshold,
    "mode": mode,
    "model_bundle_path": "models/rf_final_b2_pipeline.pkl",
    "seed": 42,
})
CONFIG_PATH.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")

# El Dockerfile usa un manifiesto de servicio pequeño, no el freeze de todo el entorno del curso.
(PROJECT_ROOT / "requirements_mlops.txt").write_text(
    "numpy>=1.24\nscikit-learn>=1.3\njoblib>=1.3\n"
    "onnxruntime>=1.16\nfastapi>=0.110\npydantic>=2.0\nuvicorn>=0.27\n",
    encoding="utf-8",
)
print(f"Datos: {df.shape}; X_test: {X_test_raw.shape}; modo del modelo: {mode}")
print(f"Clases del estimador: {getattr(model, 'classes_', 'no disponible')}")
print(f"Bundle de inferencia guardado en: {MODEL_BUNDLE_PATH}")

Modelo binario B2 cargado: c:\Users\marco\ml-demo1-profesor\models\rf_final_b2.pkl
Datos: (2200, 8); X_test: (440, 7); modo del modelo: binary
Clases del estimador: [0 1]
Bundle de inferencia guardado en: c:\Users\marco\ml-demo1-profesor\models\rf_final_b2_pipeline.pkl


## Exportar a ONNX

ONNX permite exportar el estimador y ejecutarlo con ONNX Runtime. El scaler y el clasificador se convierten como una sola tubería, evitando que el cliente tenga que replicar el preprocesamiento. El conjunto de prueba se transforma a `float32`, el tipo de entrada esperado por el modelo exportado.

Si faltan `onnx`, `onnxruntime` o `skl2onnx`, se muestra una advertencia y el notebook puede continuar usando el bundle de scikit-learn.

In [3]:
# Convertimos la tubería completa si el convertidor está disponible.
ONNX_PATH = PROJECT_ROOT / "models" / "modelo.onnx"
onnx_model = None
if ONNX_AVAILABLE:
    options = {}
    classifier = serving_model.named_steps["classifier"] if BINARY_MODEL else serving_model
    options[id(classifier)] = {"zipmap": False}
    onnx_model = convert_sklearn(
        serving_model,
        initial_types=[("input", FloatTensorType([None, len(feature_columns)]))],
        options=options,
        target_opset=17,
    )
    ONNX_PATH.write_bytes(onnx_model.SerializeToString())
    print(f"Modelo ONNX guardado: {ONNX_PATH} ({ONNX_PATH.stat().st_size / 1024:.1f} KB)")
else:
    print("Advertencia: se omite la exportación ONNX; se usará el bundle scikit-learn.")

Modelo ONNX guardado: c:\Users\marco\ml-demo1-profesor\models\modelo.onnx (207.2 KB)


## Cargar ONNX y comparar

Cargamos el archivo exportado con ONNX Runtime, ejecutamos algunas filas del test y comparamos etiquetas con el mismo pipeline de scikit-learn. La comparación verifica equivalencia funcional; no demuestra por sí sola que el modelo sea más rápido ni que esté listo para producción.

In [4]:
# Creamos la sesión de inferencia y validamos predicciones del mismo lote.
onnx_session = None
if ONNX_AVAILABLE and ONNX_PATH.exists():
    onnx_session = ort.InferenceSession(
        str(ONNX_PATH), providers=["CPUExecutionProvider"]
    )
    input_name = onnx_session.get_inputs()[0].name
    sample = X_test_raw.iloc[:5].to_numpy(dtype=np.float32)
    original_labels = serving_model.predict(sample)
    onnx_outputs = onnx_session.run(None, {input_name: sample})
    onnx_labels = np.asarray(onnx_outputs[0]).reshape(-1)
    print("Predicciones scikit-learn:", original_labels.tolist())
    print("Predicciones ONNX:", onnx_labels.tolist())
    agreement = np.mean(original_labels.astype(str) == onnx_labels.astype(str))
    print(f"Acuerdo de etiquetas: {agreement:.1%}")
    if agreement < 1.0:
        raise AssertionError("Las etiquetas de ONNX difieren de scikit-learn en el lote de control.")
else:
    print("Advertencia: ONNX Runtime o el modelo ONNX no están disponibles.")

Predicciones scikit-learn: [0, 0, 0, 0, 0]
Predicciones ONNX: [0, 0, 0, 0, 0]
Acuerdo de etiquetas: 100.0%


## Latencia: ONNX vs original

Medimos el tiempo de inferencia por observación usando la misma muestra y una llamada por fila. Se realiza una llamada de calentamiento para evitar atribuir inicialización al tiempo típico. La medición es local y depende del hardware, tamaño del lote y carga; no sustituye un benchmark representativo del servicio desplegado.

In [5]:
def medir_latencia(predict_fn, X, n=100):
    '''Mide el promedio en milisegundos por predicción individual, tras una llamada de calentamiento.'''
    X_array = np.asarray(X, dtype=np.float32)
    if len(X_array) == 0:
        raise ValueError("X debe contener al menos una fila.")
    predict_fn(X_array[:1])  # Calentamiento del runtime.
    cantidad = min(int(n), len(X_array))
    inicio = time.perf_counter()
    for fila in X_array[:cantidad]:
        predict_fn(fila.reshape(1, -1))
    return (time.perf_counter() - inicio) * 1000 / cantidad

X_latency = X_raw.iloc[:min(100, len(X_raw))].to_numpy(dtype=np.float32)
sklearn_ms = medir_latencia(serving_model.predict, X_latency)
print(f"Scikit-learn: {sklearn_ms:.4f} ms/inferencia")
onnx_ms = None
if onnx_session is not None:
    input_name = onnx_session.get_inputs()[0].name
    onnx_ms = medir_latencia(
        lambda batch: onnx_session.run(None, {input_name: batch})[0],
        X_latency,
    )
    print(f"ONNX Runtime: {onnx_ms:.4f} ms/inferencia")
    print(f"Relación sklearn/ONNX: {sklearn_ms / max(onnx_ms, 1e-12):.2f}x")

Scikit-learn: 16.2483 ms/inferencia
ONNX Runtime: 0.0380 ms/inferencia
Relación sklearn/ONNX: 427.48x


## FastAPI: el estándar para APIs

**FastAPI** permite definir endpoints tipados en Python y generar documentación interactiva. Pydantic valida los datos de entrada; el framework ofrece validación automática, documentación OpenAPI y soporte asíncrono.

Una API de ML normalmente carga el modelo una vez al arrancar, valida la solicitud, aplica exactamente el mismo preprocesamiento del entrenamiento y responde con predicción y metadatos útiles. No se debe entrenar el modelo dentro de cada solicitud.

In [6]:
%%writefile app/api.py
"""Servicio REST educativo para el clasificador de cultivos."""
import json
from pathlib import Path

import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel, Field

ROOT = Path(__file__).resolve().parents[1]
CONFIG = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))
FEATURES = CONFIG["feature_columns"]
THRESHOLD = float(CONFIG.get("threshold", 0.5))
MODE = CONFIG.get("mode", "binary")
ONNX_FILE = ROOT / "models" / "modelo.onnx"
BUNDLE_FILE = ROOT / CONFIG.get("model_bundle_path", "models/rf_final_b2_pipeline.pkl")
_session = None
_model = None
_input_name = None

class InputData(BaseModel):
    """Mediciones de entrada con límites plausibles del dominio Crop Recommendation."""
    N: float = Field(ge=0, le=140, description="Nitrógeno del suelo")
    P: float = Field(ge=5, le=145, description="Fósforo del suelo")
    K: float = Field(ge=5, le=205, description="Potasio del suelo")
    temperature: float = Field(ge=8, le=45, description="Temperatura en °C")
    humidity: float = Field(ge=14, le=100, description="Humedad relativa (%)")
    ph: float = Field(ge=3, le=10, description="pH del suelo")
    rainfall: float = Field(ge=20, le=300, description="Lluvia en mm")

app = FastAPI(title="Crop Recommendation - Clase 15", version="1.0.0")

@app.on_event("startup")
def load_model():
    """Carga ONNX Runtime cuando está disponible; usa el bundle de sklearn como respaldo."""
    global _session, _model, _input_name
    if ONNX_FILE.exists():
        try:
            import onnxruntime as ort
        except ImportError:
            import joblib
            _model = joblib.load(BUNDLE_FILE)
        else:
            _session = ort.InferenceSession(
                str(ONNX_FILE), providers=["CPUExecutionProvider"]
            )
            _input_name = _session.get_inputs()[0].name
    else:
        import joblib
        _model = joblib.load(BUNDLE_FILE)

def predict_one(values):
    """Devuelve clase y probabilidad usando el runtime activo."""
    row = np.asarray([[values[name] for name in FEATURES]], dtype=np.float32)
    if _session is not None:
        outputs = _session.run(None, {_input_name: row})
        label = np.asarray(outputs[0]).reshape(-1)[0].item()
        probability_output = outputs[1]
        if isinstance(probability_output, list) and probability_output and isinstance(probability_output[0], dict):
            probability_map = probability_output[0]
            if MODE == "binary":
                positive_probability = float(probability_map.get(1, probability_map.get("1", 0.0)))
                label = 1 if positive_probability >= THRESHOLD else 0
                confidence = positive_probability if label == 1 else 1.0 - positive_probability
            else:
                confidence = float(max(probability_map.values()))
        else:
            probabilities = np.asarray(probability_output)
            if probabilities.ndim == 1:
                probabilities = probabilities.reshape(1, -1)
            if MODE == "binary":
                positive_probability = float(probabilities[0, 1])
                label = 1 if positive_probability >= THRESHOLD else 0
                confidence = positive_probability if label == 1 else 1.0 - positive_probability
            else:
                confidence = float(np.max(probabilities[0]))
    else:
        probabilities = _model.predict_proba(row)[0]
        classes = list(_model.classes_)
        if MODE == "binary":
            positive_probability = float(probabilities[classes.index(1)])
            label = 1 if positive_probability >= THRESHOLD else 0
            confidence = positive_probability if label == 1 else 1.0 - positive_probability
        else:
            index = int(np.argmax(probabilities))
            label = classes[index]
            confidence = float(probabilities[index])
    if MODE == "binary":
        prediction = "rice" if int(label) == 1 else "no rice"
    else:
        prediction = str(label)
    return {"prediction": prediction, "class_id": int(label), "probability": confidence, "confidence": confidence}

@app.get("/health")
def health():
    """Comprueba que el servicio y el artefacto de inferencia estén cargados."""
    return {"status": "ok", "mode": MODE, "runtime": "onnx" if _session is not None else "scikit-learn"}

@app.post("/predict")
def predict(data: InputData):
    """Valida siete features y retorna la clase y su confianza."""
    return predict_one(data.model_dump())

Writing app/api.py


## Probar la API localmente

Para probar la API, ejecutar en terminal: `uvicorn app.api:app --reload`. Luego abrir [http://localhost:8000/docs](http://localhost:8000/docs).

En el aula, mostraremos la API funcionando. La ruta `/health` permite una comprobación sencilla y `/predict` acepta un JSON con las siete variables del modelo.

In [ ]:
# TestClient permite probar la aplicación dentro del notebook sin levantar un servidor.
if FASTAPI_AVAILABLE:
    from fastapi.testclient import TestClient
    # Cargamos por ruta para evitar el conflicto entre app.py (Streamlit) y el paquete app/.
    import importlib.util
    import sys
    api_path = PROJECT_ROOT / "app" / "api.py"
    api_spec = importlib.util.spec_from_file_location("class15_api", api_path)
    if api_spec is None or api_spec.loader is None:
        raise ImportError(f"No se pudo cargar el módulo de API desde {api_path}.")
    api_module = importlib.util.module_from_spec(api_spec)
    sys.modules[api_spec.name] = api_module
    api_spec.loader.exec_module(api_module)
    app = api_module.app

    sample_payload = {
        "N": float(X_raw.iloc[0]["N"]),
        "P": float(X_raw.iloc[0]["P"]),
        "K": float(X_raw.iloc[0]["K"]),
        "temperature": float(X_raw.iloc[0]["temperature"]),
        "humidity": float(X_raw.iloc[0]["humidity"]),
        "ph": float(X_raw.iloc[0]["ph"]),
        "rainfall": float(X_raw.iloc[0]["rainfall"]),
    }
    with TestClient(app) as client:
        health_response = client.get("/health")
        valid_response = client.post("/predict", json=sample_payload)
        invalid_payload = {**sample_payload, "humidity": 1000}
        invalid_response = client.post("/predict", json=invalid_payload)

    print("GET /health:", health_response.status_code, health_response.json())
    print("POST /predict válido:", valid_response.status_code, valid_response.json())
    print("POST /predict inválido:", invalid_response.status_code)
    assert health_response.status_code == 200
    assert valid_response.status_code == 200
    assert invalid_response.status_code == 422
else:
    print("Advertencia: instala FastAPI, Pydantic y httpx para ejecutar estas pruebas.")

## Pydantic: validación de datos

**Pydantic** valida y convierte los datos usando anotaciones de tipos. Por ejemplo, `Field(ge=0, le=140)` exige que el nitrógeno esté dentro del rango definido; los demás campos tienen límites coherentes con el dominio y el dataset de la práctica.

Si el usuario envía un valor fuera de rango o un tipo incompatible, FastAPI devuelve error **422** automáticamente. La validación reduce errores de entrada, pero no reemplaza la revisión del dominio ni el monitoreo de datos fuera de distribución.

## Docker: contenerización

**Docker** empaqueta la aplicación y sus dependencias en una imagen reproducible. Aporta aislamiento, portabilidad y una base común para escalar, aunque no sustituye controles de seguridad ni monitoreo.

Estructura: elegir una imagen base; fijar e instalar dependencias; copiar el código y los artefactos; exponer el puerto; y definir el comando de inicio. El `requirements.txt` raíz de este proyecto es un freeze del entorno de curso, no una lista apropiada para un servicio mínimo. Por eso generamos `requirements_mlops.txt`, con dependencias de serving.

In [ ]:
%%writefile Dockerfile
FROM python:3.10-slim
WORKDIR /app
COPY requirements_mlops.txt .
RUN pip install --no-cache-dir -r requirements_mlops.txt
COPY app/ ./app/
COPY models/ ./models/
COPY config.json ./config.json
EXPOSE 8000
CMD ["uvicorn", "app.api:app", "--host", "0.0.0.0", "--port", "8000"]

## Comandos Docker

Ejecuta estos comandos desde la raíz del proyecto, donde se encuentran el `Dockerfile`, `app/`, `models/`, `config.json` y `requirements_mlops.txt`.

| Acción | Comando |
|---|---|
| Construir imagen | `docker build -t crop-api:1.0 .` |
| Ejecutar contenedor | `docker run --name crop-api -p 8000:8000 crop-api:1.0` |
| Listar contenedores | `docker ps` |
| Consultar logs | `docker logs crop-api` |
| Detener | `docker stop crop-api` |
| Eliminar contenedor detenido | `docker rm crop-api` |
| Eliminar imagen | `docker rmi crop-api:1.0` |

En el aula, mostraremos el contenedor funcionando. Para una entrega, no incluyas secretos ni datos sensibles en la imagen.

## Profiling: midiendo recursos

Las métricas básicas de un servicio incluyen **latencia** (tiempo por solicitud), **RAM** (memoria residente), **CPU** (tiempo de procesador) y **tamaño del modelo**. Para comparar, fija hardware, lote, calentamiento y número de repeticiones.

La latencia se expresa aquí en milisegundos por inferencia. `psutil` permite consultar el proceso local; el tamaño en disco no equivale a la memoria ocupada al cargar el modelo. En despliegues reales, monitorea percentiles (p50/p95/p99), no solo el promedio.

In [ ]:
def medir_ram():
    '''Retorna el uso de RAM residente del proceso actual en MB, si psutil está disponible.'''
    if not PSUTIL_AVAILABLE:
        return None
    return psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)

# Reportamos tamaño de los artefactos y la RAM del proceso del notebook.
bundle_mb = MODEL_BUNDLE_PATH.stat().st_size / (1024 ** 2)
print(f"Tamaño del bundle scikit-learn: {bundle_mb:.2f} MB")
if ONNX_PATH.exists():
    print(f"Tamaño del modelo ONNX: {ONNX_PATH.stat().st_size / (1024 ** 2):.2f} MB")
ram_mb = medir_ram()
print(f"RAM residente del proceso: {ram_mb:.2f} MB" if ram_mb is not None else "RAM: instala psutil para medirla.")

## Green AI: sostenibilidad energética

Una estimación simplificada usa:

- **Energía (kWh) = potencia (kW) × tiempo (horas)**.
- **Huella (kg CO₂e) = energía (kWh) × factor de emisión (kg CO₂e/kWh)**.

Para el ejercicio usamos un factor de emisión ilustrativo de Ecuador de **0.4 kg CO₂e/kWh**; es una aproximación docente, no un valor universal ni necesariamente vigente para cada región o proveedor. Las estrategias incluyen modelos ligeros, cuantización, ONNX Runtime y elección de un batch size adecuado. En CD 16 y P3-7 profundizarán.

In [ ]:
# Estimamos energía usando la latencia local; la potencia del servidor es una hipótesis didáctica.
potencia_kw = 0.1
factor_emision = 0.4
latencia_ms = onnx_ms if onnx_ms is not None else sklearn_ms
energia_por_inferencia_kwh = potencia_kw * (latencia_ms / 1000 / 3600)
huella_por_inferencia_kg = energia_por_inferencia_kwh * factor_emision
inferencias_anuales = 1_000_000
huella_anual_kg = huella_por_inferencia_kg * inferencias_anuales
print(f"Energía estimada por inferencia: {energia_por_inferencia_kwh:.3e} kWh")
print(f"Huella estimada por inferencia: {huella_por_inferencia_kg:.3e} kg CO2e")
print(f"Huella ilustrativa para 1 millón: {huella_anual_kg:.4f} kg CO2e")
print("La estimación no incluye infraestructura ociosa, red, almacenamiento ni refrigeración.")

## Conexión con P3

| Entregable | Lo visto hoy |
|---|---|
| P3-1: ONNX | Exportación y comparación de predicciones |
| P3-2: API | Endpoint `/predict` con FastAPI |
| P3-3: Pydantic | Validación tipada y respuesta 422 |
| P3-4: Dockerfile | Imagen con dependencias y artefactos |
| P3-5: contenedor | Ejecución reproducible del servicio |
| P3-6: profiling | Latencia, RAM y tamaño |

Todo lo que vimos hoy lo aplicarán en P3. Documenten el entorno, el contrato de entrada, los límites conocidos y las pruebas; el despliegue no convierte automáticamente una predicción en una decisión confiable.

## Conclusiones

1. **MLOps** conecta datos, modelos, despliegue y monitoreo en un ciclo reproducible.
2. **ONNX** facilita la interoperabilidad; la equivalencia debe comprobarse.
3. **FastAPI y Pydantic** ofrecen serving tipado y validación automática.
4. **Docker** empaqueta la aplicación con dependencias explícitas.
5. **Profiling** mide rendimiento y recursos con un protocolo comparable.
6. **Green AI** hace visibles los costos energéticos y sus supuestos.
7. El paso a producción exige pruebas, observabilidad, documentación y revisión de riesgos.

**Próxima clase:** CD 16 (profiling, Green AI y cierre del Bloque 3).

In [ ]:
# Mensaje final y artefactos generados por la ejecución.
print("Clase 15 completada. Revisa app/api.py, Dockerfile y models/modelo.onnx")